# Unified Multi-Output Weather Model (ESP32)

This notebook trains a single Dense (MLP) model to simultaneously predict precipitation and evapotranspiration using the expanded weather dataset.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Flatten
from sklearn.metrics import mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

I0000 00:00:1777800570.784690  123765 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## 2. Load Dataset

In [2]:
df = pd.read_csv('historical_weather_data.csv')
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values('time').reset_index(drop=True)
print(f'Loaded {len(df)} days of historical weather data.')

Loaded 1461 days of historical weather data.


## 3. Feature Engineering

In [3]:
weather_cols = ['temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'et0_fao_evapotranspiration', 'shortwave_radiation_sum', 'soil_moisture_0_to_7cm', 'relative_humidity_2m', 'vapor_pressure_deficit', 'wind_speed_10m']

for col in weather_cols:
    for i in range(1, 4):
        df[f'{col}_lag_{i}'] = df[col].shift(i)

df['target_precipitation'] = df['precipitation'].shift(-1)
df['target_evapotranspiration'] = df['evapotranspiration'].shift(-1)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

X_3d = np.zeros((len(df), 3, len(weather_cols)))
for i, col in enumerate(weather_cols):
    X_3d[:, 0, i] = df[f'{col}_lag_3']
    X_3d[:, 1, i] = df[f'{col}_lag_2']
    X_3d[:, 2, i] = df[f'{col}_lag_1']

# Unified multi-output target array: shape (N, 2)
y = np.column_stack((df['target_precipitation'].values, df['target_evapotranspiration'].values))

print(f'Input Shape: {X_3d.shape}')
print(f'Target Shape: {y.shape}')


Input Shape: (1457, 3, 9)
Target Shape: (1457, 2)


## 4. Train/Test Split & Normalization

In [4]:
split_idx = int(len(df) * 0.8)

X_train, X_test = X_3d[:split_idx], X_3d[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

X_mean = X_train.mean(axis=0)
X_std = X_train.std(axis=0)
X_std[X_std == 0] = 1e-6

X_train_scaled = (X_train - X_mean) / X_std
X_test_scaled = (X_test - X_mean) / X_std

np.save('X_mean_unified.npy', X_mean)
np.save('X_std_unified.npy', X_std)
print(f'Training on {len(X_train)} days, Testing on {len(X_test)} days.')


Training on 1165 days, Testing on 292 days.


## 5. Unified Model Architecture

In [5]:
model = Sequential([
    Input(shape=(3, len(weather_cols))),
    Flatten(),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(2) # 2 Outputs: [Precipitation, ET0]
])

model.compile(optimizer='adam', loss='mse')
print(model.summary())


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 27)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,458 (5.70 KB)

 Trainable params: 1,458 (5.70 KB)

 Non-trainable params: 0 (0.00 B)

None


## 6. Train Model

In [6]:
print("Training Unified Model...")
model.fit(X_train_scaled, y_train, epochs=60, batch_size=16, verbose=0, validation_data=(X_test_scaled, y_test))
print("Training complete.")


Training Unified Model...


Training complete.


## 7. Performance Evaluation

In [7]:
preds = model.predict(X_test_scaled, verbose=0)

# Slice outputs
pred_precip = preds[:, 0]
pred_evapotranspiration = preds[:, 1]
y_test_precip = y_test[:, 0]
y_test_evapotranspiration = y_test[:, 1]

mae_p = mean_absolute_error(y_test_precip, pred_precip)
r2_p = r2_score(y_test_precip, pred_precip)

mae_e = mean_absolute_error(y_test_evapotranspiration, pred_evapotranspiration)
r2_e = r2_score(y_test_evapotranspiration, pred_evapotranspiration)

print("========== PRECIPITATION (Unified) ==========")
print(f"MAE: {mae_p:.3f} mm  | R²: {r2_p:.3f}")
print("\n========== EVAPOTRANSPIRATION (Unified) ==========")
print(f"MAE: {mae_e:.3f} mm  | R²: {r2_e:.3f}")


========== PRECIPITATION (Unified) ==========
MAE: 2.918 mm  | R²: -0.052

========== EVAPOTRANSPIRATION (Unified) ==========
MAE: 0.625 mm  | R²: -0.266


## 8. Export to TFLite

In [8]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('unified_weather_model.tflite', 'wb') as f:
    f.write(tflite_model)
print(f'Saved unified_weather_model.tflite ({len(tflite_model)} bytes)')


INFO:tensorflow:Assets written to: /tmp/tmpshn43hm0/assets


INFO:tensorflow:Assets written to: /tmp/tmpshn43hm0/assets


Saved artifact at '/tmp/tmpshn43hm0'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 3, 9), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  140548338236928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140548338379840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140548338327840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140548338328800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140548338330720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140548338329520: TensorSpec(shape=(), dtype=tf.resource, name=None)


Saved unified_weather_model.tflite (8832 bytes)


W0000 00:00:1777800587.264420  123765 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1777800587.264439  123765 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1777800587.264948  123765 reader.cc:83] Reading SavedModel from: /tmp/tmpshn43hm0
I0000 00:00:1777800587.265637  123765 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1777800587.265658  123765 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpshn43hm0
I0000 00:00:1777800587.269474  123765 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1777800587.270044  123765 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1777800587.295194  123765 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpshn43hm0
I0000 00:00:1777800587.303482  123765 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 38543 microseconds.
I0000 00:00:1777800587.332637  123765

## 9. Generate C Array

In [9]:
def convert_tflite_to_c_array(tflite_path, c_file_path, array_name):
    with open(tflite_path, 'rb') as f:
        tflite_content = f.read()

    hex_array = [f'0x{b:02x}' for b in tflite_content]
    
    with open(c_file_path, 'w') as f:
        f.write(f'#include "{array_name}.h"\n\n')
        f.write(f'const unsigned char {array_name}[] = {{\n')
        for i in range(0, len(hex_array), 12):
            f.write('  ' + ', '.join(hex_array[i:i+12]) + ',\n')
        f.write('};\n\n')
        f.write(f'const int {array_name}_len = {len(hex_array)};\n')

    h_file_path = c_file_path.replace('.cc', '.h')
    with open(h_file_path, 'w') as f:
        f.write(f'#ifndef {array_name.upper()}_H\n')
        f.write(f'#define {array_name.upper()}_H\n\n')
        f.write(f'extern const unsigned char {array_name}[];\n')
        f.write(f'extern const int {array_name}_len;\n\n')
        f.write(f'#endif // {array_name.upper()}_H\n')
        
    print(f'Generated {c_file_path} and {h_file_path}')

convert_tflite_to_c_array('unified_weather_model.tflite', 'unified_weather_model.cc', 'unified_weather_model_tflite')


Generated unified_weather_model.cc and unified_weather_model.h
